# 合成路线 → 结构描述 → CrystalFormer 晶体生成

选择已安装依赖的内核（如 **route-crystal** 或 **highk_bec**），按顺序运行。第一阶段保留原来的 ELM 化学式生成、本地 RAG、DeepSeek 路线和英文结构描述生成，停止于描述。

第二阶段采用 `../CrystalFormer_固定化学式生成.ipynb` 的原版 CrystalFormer-CSP：读取每个结构假设对应的化学式与第一个候选空间群，加载 `alex20s/csp/epoch_046000.pkl` 真实权重，生成晶格、Wyckoff 占位和坐标，不手工指定晶格常数。

**CrystalFormer 的生成条件是化学式和空间群。** 描述中的配位、层状和连接方式只保存，不作为文本条件输入；通过检查的结构仍是未弛豫候选。

第一阶段 API 和并发设置仍取同目录 `config.json`。第二阶段采样参数取 `crystalformer` 节，默认每个结构假设采样 8 个，批量 8，温度 1.0、top_p=1.0。Windows 沿用参考 Notebook 的 JAX CPU，采样包含模型加载和编译时间。

新批次保持 `RESUME_RUN=None`；已有兼容配置的描述批次可以填入 `RESUME_RUN` 复用。第一阶段仍使用 `low` 推理强度和 4 个化学式并发；CrystalFormer 阶段逐个结构假设运行。


## 1. 仅生成合成路线与结构描述

In [ ]:
from pathlib import Path
import sys
import importlib

BUNDLE = Path.cwd() if Path.cwd().name == "crystal_pipeline" else Path.cwd() / "crystal_pipeline"
sys.path.insert(0, str(BUNDLE))
from modules import chemistry, routes, crystal_generation, deepseek_api, configuration
import pipeline

for module in (deepseek_api, configuration, chemistry, routes, crystal_generation, pipeline):
    importlib.reload(module)
from pipeline import run

RESUME_RUN = None
result_dir = run(BUNDLE / "config.json", resume_run=RESUME_RUN, stage="descriptions")

In [ ]:
from IPython.display import display, Markdown

display(Markdown((result_dir / "descriptions.md").read_text(encoding="utf-8")))

## 2. 用 CrystalFormer 按化学式和空间群生成 CIF

此单元格读取第一阶段已保存的描述，使用每个有效结构假设的化学式和第一个候选空间群编号。第二阶段不请求 API。

每个假设默认原始采样 8 个，调用官方 `scripts/awl2struct.py` 展开对称操作并筛选组成，再将 CIF 回读，检查实际空间群和原子重叠。全部通过检查的候选保存为 `candidate_001.cif` 等，第一份另存为 `candidate.cif` 供汇总报告使用；不按能量排序。没有通过检查的样本时记录失败并继续其他假设，不改换目标空间群。

结果写入该批次的 `crystals_crystalformer/`。每个假设保留 `description.txt`、原始采样 CSV、组成筛选 CSV、`sampling.log`、`conversion.log`、候选 CIF 和 `metadata.json`。旧结果保留。采样数不等于最终通过检查的结构数，也不表示满足描述中的配位和拓扑或已具有热力学稳定性。


In [ ]:
from modules import crystalformer_crystal
importlib.reload(crystalformer_crystal)

result_dir = run(
    BUNDLE / "config.json",
    resume_run=result_dir,
    stage="crystals",
    crystal_model="crystalformer",
)


In [ ]:
report_file = result_dir / "descriptions.md"
if (result_dir / "results.md").is_file():
    report_file = result_dir / "results.md"

display(Markdown(report_file.read_text(encoding="utf-8")))